# Lesson 0B: Gymnasium Setup - Practical

## Introduction

Now that we understand RL concepts theoretically, let's learn to **use the tools** that every RL practitioner uses: the Gymnasium environment API.

**Gymnasium** (formerly OpenAI Gym) is the standard library for defining and interacting with RL environments. Whether you're working with simple GridWorlds or complex robotic simulations, Gymnasium provides a consistent interface.

In this lesson, we'll:
1. Install and import Gymnasium
2. Understand the environment API: `reset()`, `step()`, `render()`
3. Explore built-in environments (CartPole, MountainCar, Atari, etc.)
4. Implement a custom environment from scratch
5. Learn visualization techniques for RL
6. Master debugging tips for common issues

By the end, you'll be able to use **any** Gymnasium environment and build your own.

## Table of Contents

1. [Required Libraries](#required-libraries)
2. [Gymnasium Installation](#installation)
3. [The Core API: reset(), step(), render()](#core-api)
4. [Built-in Environments](#builtin-envs)
5. [Custom Environment Implementation](#custom-env)
6. [Visualization Techniques](#visualization)
7. [Debugging RL Agents](#debugging)
8. [Best Practices](#best-practices)

<a name="required-libraries"></a>
## Required Libraries

| Library | Purpose |
|---------|----------|
| Gymnasium | RL environment API (replaces OpenAI Gym) |
| NumPy | Numerical operations |
| Matplotlib | Visualization |
| Seaborn | Statistical plots |
| IPython | Jupyter display utilities |

<a name="installation"></a>
## Gymnasium Installation

Gymnasium is the modern successor to OpenAI Gym. Some key differences:
- Built-in support for modern Python type hints
- Better environment wrappers and utilities
- More active development and maintenance
- Compatible with latest RL libraries (Stable-Baselines3, RLlib, etc.)

In [ ]:
import subprocess
import sys

# Install gymnasium with all extras
try:
    import gymnasium
    print(f"✅ Gymnasium already installed (version {gymnasium.__version__})")
except ImportError:
    print("📦 Installing Gymnasium...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "gymnasium[atari,classic_control,box2d]", "-q"])
    import gymnasium
    print(f"✅ Gymnasium installed (version {gymnasium.__version__})")

# Import all needed libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gymnasium as gym
from gymnasium import Env, spaces
from typing import Tuple, Dict, Any, Optional
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Settings
np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("\n✅ All libraries loaded!")
print(f"NumPy: {np.__version__}")
print(f"Gymnasium: {gym.__version__}")

<a name="core-api"></a>
## The Core API: reset(), step(), render()

Every Gymnasium environment follows the **same interface**. This is the power of the standard!

### reset()
```python
obs, info = env.reset()
```
- **Purpose**: Initialize the environment for a new episode
- **Returns**:
  - `obs`: Initial observation (state)
  - `info`: Optional metadata (dict)

### step(action)
```python
obs, reward, terminated, truncated, info = env.step(action)
```
- **Purpose**: Execute one action in the environment
- **Input**: `action` from the action space
- **Returns**:
  - `obs`: New observation (state)
  - `reward`: Scalar reward
  - `terminated`: Whether episode ended (goal/failure)
  - `truncated`: Whether episode ended (time limit)
  - `info`: Optional metadata

### render()
```python
env.render()  # Display the environment
```
- **Purpose**: Visualize the current state
- **Note**: Some environments support `mode='rgb_array'` to return pixels instead of displaying

In [ ]:
# Create a CartPole environment
env = gym.make('CartPole-v1', render_mode='rgb_array')

print("Environment created!")
print(f"\nEnvironment info:")
print(f"  ID: CartPole-v1")
print(f"  Observation space: {env.observation_space}")
print(f"  Action space: {env.action_space}")
print(f"\nObservation space details:")
print(f"  Type: {type(env.observation_space)}")
print(f"  Shape: {env.observation_space.shape}")
print(f"  Bounds: [{env.observation_space.low}, {env.observation_space.high}]")
print(f"\nAction space details:")
print(f"  Type: {type(env.action_space)}")
print(f"  Num actions: {env.action_space.n}")

In [ ]:
# Demonstrate reset() and step()
print("\n" + "="*60)
print("Demonstrating reset() and step()")
print("="*60)

# Reset
obs, info = env.reset(seed=42)
print(f"\nAfter reset():")
print(f"  Observation: {obs}")
print(f"    - Cart position: {obs[0]:.3f}")
print(f"    - Cart velocity: {obs[1]:.3f}")
print(f"    - Pole angle: {obs[2]:.3f} rad")
print(f"    - Pole angular velocity: {obs[3]:.3f} rad/s")
print(f"  Info: {info}")

# Take 5 random actions
print(f"\nTaking 5 random actions:")
print("-" * 60)

for step_num in range(5):
    action = env.action_space.sample()  # Random action
    obs, reward, terminated, truncated, info = env.step(action)
    
    print(f"\nStep {step_num + 1}:")
    print(f"  Action: {action} ({'push left' if action == 0 else 'push right'})")
    print(f"  Observation: {np.round(obs, 3)}")
    print(f"  Reward: {reward}")
    print(f"  Terminated: {terminated} (pole fell or went out of bounds)")
    print(f"  Truncated: {truncated} (time limit reached)")
    print(f"  Done: {terminated or truncated}")
    
    if terminated or truncated:
        print(f"\n✅ Episode ended!")
        break

env.close()

<a name="builtin-envs"></a>
## Built-in Environments

Gymnasium provides many classic control tasks and games:

### Classic Control
- **CartPole**: Balance a pole on a moving cart
- **MountainCar**: Drive a car up a mountain
- **Pendulum**: Swing up and balance an inverted pendulum
- **Acrobot**: Swing up and balance a 2-link robot arm

### Box2D (Physics Simulation)
- **LunarLander**: Land a spacecraft on the moon
- **BipedalWalker**: Make a 2-legged robot walk
- **CarRacing**: Autonomous race car

### Atari Games
- **Pong**, **Breakout**, **SpaceInvaders**, **Tetris**, etc.
- Requires: `gymnasium[atari]`

Let's explore a few:

In [ ]:
# List available classic control environments
print("Available Classic Control Environments:")
print("-" * 60)

classic_control_envs = [
    'CartPole-v1',
    'MountainCar-v0',
    'Pendulum-v1',
    'Acrobot-v1',
]

for env_id in classic_control_envs:
    try:
        env = gym.make(env_id, render_mode=None)
        print(f"\n{env_id}:")
        print(f"  Observation: {env.observation_space}")
        print(f"  Action: {env.action_space}")
        env.close()
    except Exception as e:
        print(f"\n{env_id}: ❌ Not available ({str(e)[:50]}...)")

In [ ]:
# Run a full episode and collect statistics
env = gym.make('CartPole-v1')

obs, _ = env.reset(seed=42)
total_reward = 0
step_count = 0
observations = [obs.copy()]

print("Running a random policy episode...\n")

for step_num in range(500):
    action = env.action_space.sample()  # Random action
    obs, reward, terminated, truncated, _ = env.step(action)
    observations.append(obs.copy())
    
    total_reward += reward
    step_count += 1
    
    if terminated or truncated:
        break

observations = np.array(observations)

print(f"Episode Summary:")
print(f"  Total steps: {step_count}")
print(f"  Total reward: {total_reward}")
print(f"  Average reward per step: {total_reward / step_count:.3f}")
print(f"\nObservation ranges over episode:")
print(f"  Cart position: [{observations[:, 0].min():.3f}, {observations[:, 0].max():.3f}]")
print(f"  Cart velocity: [{observations[:, 1].min():.3f}, {observations[:, 1].max():.3f}]")
print(f"  Pole angle: [{observations[:, 2].min():.3f}, {observations[:, 2].max():.3f}]")
print(f"  Pole velocity: [{observations[:, 3].min():.3f}, {observations[:, 3].max():.3f}]")

env.close()

In [ ]:
# Visualize observation dynamics
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

steps = np.arange(len(observations))

axes[0, 0].plot(steps, observations[:, 0], linewidth=2)
axes[0, 0].set_ylabel('Cart Position (m)', fontsize=11, fontweight='bold')
axes[0, 0].set_title('CartPole Dynamics', fontsize=12, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(steps, observations[:, 1], linewidth=2, color='orange')
axes[0, 1].set_ylabel('Cart Velocity (m/s)', fontsize=11, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(steps, observations[:, 2], linewidth=2, color='green')
axes[1, 0].set_ylabel('Pole Angle (rad)', fontsize=11, fontweight='bold')
axes[1, 0].set_xlabel('Step', fontsize=11, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(steps, observations[:, 3], linewidth=2, color='red')
axes[1, 1].set_ylabel('Pole Velocity (rad/s)', fontsize=11, fontweight='bold')
axes[1, 1].set_xlabel('Step', fontsize=11, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

<a name="custom-env"></a>
## Custom Environment Implementation

The Gymnasium API is a **contract** that any environment must follow. By implementing it, your custom environment works with any RL algorithm!

Here's how to build a custom environment:

```python
class MyEnv(gym.Env):
    metadata = {'render_modes': ['human', 'rgb_array'], 'render_fps': 30}
    
    def __init__(self, render_mode=None):
        self.observation_space = spaces.Box(...)  # Define observation space
        self.action_space = spaces.Discrete(...)  # Define action space
        self.render_mode = render_mode
    
    def reset(self, seed=None, options=None):
        """Initialize episode. Return obs, info."""
        super().reset(seed=seed)
        # ... your reset logic ...
        return obs, info
    
    def step(self, action):
        """Execute action. Return obs, reward, terminated, truncated, info."""
        # ... your step logic ...
        return obs, reward, terminated, truncated, info
    
    def render(self):
        """Visualize environment if render_mode is set."""
        pass
```

Let's implement a **TreasureHunt** environment:

In [ ]:
class TreasureHuntEnv(gym.Env):
    """
    Simple 1D treasure hunt environment.
    
    Agent starts at position 0 and must find treasure at position 5.
    Actions: 0=left, 1=right
    Reward: +1 for reaching treasure, -0.1 per step
    """
    
    metadata = {'render_modes': ['human', 'rgb_array'], 'render_fps': 4}
    
    def __init__(self, render_mode=None, treasure_position=5, max_steps=20):
        # Spaces
        self.observation_space = spaces.Box(
            low=0, high=10, shape=(1,), dtype=np.float32
        )
        self.action_space = spaces.Discrete(2)  # 0=left, 1=right
        
        # Environment parameters
        self.treasure_position = treasure_position
        self.max_steps = max_steps
        self.render_mode = render_mode
        
        # State
        self.agent_position = 0
        self.step_count = 0
    
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.agent_position = 0
        self.step_count = 0
        observation = self._get_obs()
        info = {'treasure_position': self.treasure_position}
        return observation, info
    
    def step(self, action):
        # Execute action
        if action == 0:  # Left
            self.agent_position = max(0, self.agent_position - 1)
        elif action == 1:  # Right
            self.agent_position = min(10, self.agent_position + 1)
        
        self.step_count += 1
        
        # Calculate reward
        if self.agent_position == self.treasure_position:
            reward = 1.0
            terminated = True
        else:
            reward = -0.1
            terminated = False
        
        # Check time limit
        truncated = self.step_count >= self.max_steps
        
        observation = self._get_obs()
        info = {
            'treasure_position': self.treasure_position,
            'distance': abs(self.agent_position - self.treasure_position)
        }
        
        return observation, reward, terminated, truncated, info
    
    def _get_obs(self):
        return np.array([self.agent_position], dtype=np.float32)
    
    def render(self):
        if self.render_mode == 'human':
            # Print ASCII art
            line = ['.' for _ in range(11)]
            line[int(self.agent_position)] = 'A'
            line[self.treasure_position] = 'T'
            print(' '.join(line))
            print('0 1 2 3 4 5 6 7 8 9 10')
    
    def close(self):
        pass


print("✅ TreasureHuntEnv created!")

# Test it
env = TreasureHuntEnv(render_mode='human')
obs, info = env.reset(seed=42)

print(f"\nInitial observation: {obs}")
print(f"Initial info: {info}")
print(f"\nEnvironment visual:")
env.render()

print(f"\nTaking 3 actions (all right):")
for i in range(3):
    obs, reward, terminated, truncated, info = env.step(1)  # Go right
    print(f"\nStep {i+1}: Position {obs[0]:.0f}, Reward {reward}, Distance {info['distance']:.0f}")
    env.render()

env.close()

<a name="visualization"></a>
## Visualization Techniques

Visualizing RL experiments is crucial for understanding what agents are learning.

### Techniques
1. **Render the environment** at regular intervals
2. **Plot learning curves** (reward over time)
3. **Heatmaps of state visitation** (which states get explored)
4. **Policy visualization** (arrows showing action direction)
5. **Value function heatmaps** (expected reward at each state)

In [ ]:
# Train a simple agent on TreasureHunt and visualize learning
env = TreasureHuntEnv(render_mode=None, treasure_position=5)

# Simple Q-Learning agent
class QLearningAgent:
    def __init__(self, n_states, n_actions, learning_rate=0.1, gamma=0.99, epsilon=0.1):
        self.q_table = np.zeros((n_states, n_actions))
        self.learning_rate = learning_rate
        self.gamma = gamma
        self.epsilon = epsilon
    
    def act(self, state, training=True):
        if training and np.random.random() < self.epsilon:
            return np.random.randint(len(self.q_table[0]))
        return np.argmax(self.q_table[state])
    
    def learn(self, state, action, reward, next_state, done):
        if done:
            target = reward
        else:
            target = reward + self.gamma * np.max(self.q_table[next_state])
        
        self.q_table[state, action] += self.learning_rate * (target - self.q_table[state, action])

# Training
agent = QLearningAgent(n_states=11, n_actions=2)
episode_rewards = []
episode_lengths = []

for episode in range(100):
    obs, _ = env.reset()
    state = int(obs[0])
    total_reward = 0
    steps = 0
    
    for _ in range(env.max_steps):
        action = agent.act(state, training=True)
        obs, reward, terminated, truncated, _ = env.step(action)
        next_state = int(obs[0])
        
        agent.learn(state, action, reward, next_state, terminated or truncated)
        
        total_reward += reward
        steps += 1
        state = next_state
        
        if terminated or truncated:
            break
    
    episode_rewards.append(total_reward)
    episode_lengths.append(steps)

env.close()

# Visualize learning
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Learning curve
ax = axes[0]
ax.plot(episode_rewards, alpha=0.6, label='Episode Reward')
ma = np.convolve(episode_rewards, np.ones(10)/10, mode='valid')
ax.plot(range(9, len(episode_rewards)), ma, linewidth=2.5, label='10-episode MA', color='red')
ax.set_xlabel('Episode', fontsize=11, fontweight='bold')
ax.set_ylabel('Total Reward', fontsize=11, fontweight='bold')
ax.set_title('Learning Progress', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Q-table visualization
ax = axes[1]
im = ax.imshow(agent.q_table, aspect='auto', cmap='RdYlGn')
ax.set_xlabel('Action', fontsize=11, fontweight='bold')
ax.set_ylabel('Position', fontsize=11, fontweight='bold')
ax.set_title('Learned Q-Values', fontsize=12, fontweight='bold')
ax.set_xticks([0, 1])
ax.set_xticklabels(['Left', 'Right'])
plt.colorbar(im, ax=ax, label='Q-Value')

plt.tight_layout()
plt.show()

print(f"Training summary:")
print(f"  Initial avg reward: {np.mean(episode_rewards[:10]):.3f}")
print(f"  Final avg reward: {np.mean(episode_rewards[-10:]):.3f}")
print(f"  Initial avg steps: {np.mean(episode_lengths[:10]):.1f}")
print(f"  Final avg steps: {np.mean(episode_lengths[-10:]):.1f}")

<a name="debugging"></a>
## Debugging RL Agents

### Common Issues and Solutions

#### Issue 1: Agent doesn't learn (constant reward)
**Likely causes:**
- Learning rate too high/low (try 0.01-0.1)
- Exploration too high (ε should decay over time)
- Reward signal is too sparse (try reward shaping)

#### Issue 2: Agent gets stuck in local optima
**Solutions:**
- Increase exploration (higher ε)
- Use different initialization
- Try a different algorithm

#### Issue 3: Training is unstable
**Causes:**
- Learning rate too high
- Large reward variance
- Target network updates too frequent (for DQN-style methods)

#### Issue 4: Environment resets incorrectly
**Check:**
- `reset()` returns correct initial observation
- `reset()` resets all state variables
- Multiple `reset()` calls give independent episodes

### Debugging Checklist

In [ ]:
# Debugging checklist and utilities
print("\n" + "="*60)
print("RL Debugging Checklist")
print("="*60)

env = gym.make('CartPole-v1')

checks = [
    ("Observation space is Box or Discrete", True),
    ("Action space is Box or Discrete", True),
    ("reset() returns (obs, info) tuple", None),
    ("step() returns 5-tuple", None),
    ("reward is scalar", None),
    ("terminated is bool", None),
    ("truncated is bool", None),
    ("obs dtype matches observation_space", None),
    ("obs in observation_space bounds", None),
]

# Run checks
obs, info = env.reset()
print(f"\nObservation space: {env.observation_space} ✅")
print(f"Action space: {env.action_space} ✅")
print(f"\nreset() output:")
print(f"  obs: {type(obs)} ✅")
print(f"  info: {type(info)} ✅")

for _ in range(3):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)

print(f"\nstep() output:")
print(f"  obs type: {type(obs)}, shape: {obs.shape}, dtype: {obs.dtype} ✅")
print(f"  obs in bounds: {env.observation_space.contains(obs)} ✅")
print(f"  reward type: {type(reward)}, value: {reward} ✅")
print(f"  terminated type: {type(terminated)}, value: {terminated} ✅")
print(f"  truncated type: {type(truncated)}, value: {truncated} ✅")
print(f"  info type: {type(info)} ✅")

print("\n✅ All checks passed! Environment is well-formed.")

env.close()

<a name="best-practices"></a>
## Best Practices

### Environment Design
1. **Use the standard Gymnasium API** — This makes your environment compatible with all RL algorithms
2. **Keep observations normalized** — Typically in [-1, 1] or [0, 1] for better learning
3. **Design reward functions carefully**
   - Sparse rewards (only at goal) are harder to learn from
   - Shaped rewards (intermediate rewards) can help
   - Avoid reward leakage (agent should not exploit unintended rewards)
4. **Provide informative info dicts** — Extra metadata helps debugging and analysis

### Agent Development
1. **Start simple** — Test on easy environments first (CartPole before Atari)
2. **Monitor learning** — Plot learning curves and sample trajectories regularly
3. **Hyperparameter tuning** — Learning rate, exploration rate, discount factor matter a lot
4. **Seed for reproducibility** — Always set seeds for deterministic debugging
5. **Test your agent** — Evaluate on new seeds/conditions after training

### Code Organization
1. **Separate environment from agent** — Makes code reusable
2. **Log everything** — Rewards, actions, losses for analysis
3. **Version your code** — Track which version trained which model
4. **Save checkpoints** — Don't lose trained models mid-experiment

In [ ]:
# Template for a well-structured RL training script

print("\n" + "="*60)
print("RL Training Template")
print("="*60)

template = '''# Well-structured RL training

class RLTrainer:
    """Template for training RL agents."""
    
    def __init__(self, env_id, agent_class, hyperparams):
        self.env = gym.make(env_id)
        self.agent = agent_class(**hyperparams)
        self.history = {'rewards': [], 'lengths': [], 'losses': []}
    
    def train(self, n_episodes, log_interval=10):
        """Main training loop."""
        for episode in range(n_episodes):
            obs, _ = self.env.reset(seed=episode)
            episode_reward = 0
            episode_length = 0
            
            done = False
            while not done:
                # Agent chooses action
                action = self.agent.get_action(obs)
                
                # Environment responds
                obs, reward, terminated, truncated, info = self.env.step(action)
                done = terminated or truncated
                
                # Agent learns
                self.agent.learn(reward, done)
                
                episode_reward += reward
                episode_length += 1
            
            # Track progress
            self.history['rewards'].append(episode_reward)
            self.history['lengths'].append(episode_length)
            
            # Logging
            if (episode + 1) % log_interval == 0:
                avg_reward = np.mean(self.history['rewards'][-log_interval:])
                print(f"Episode {episode + 1}: Avg Reward = {avg_reward:.2f}")
    
    def evaluate(self, n_episodes=10):
        """Test trained agent without exploration."""
        returns = []
        for _ in range(n_episodes):
            obs, _ = self.env.reset()
            total_reward = 0
            done = False
            
            while not done:
                action = self.agent.get_action(obs, training=False)
                obs, reward, terminated, truncated, _ = self.env.step(action)
                done = terminated or truncated
                total_reward += reward
            
            returns.append(total_reward)
        
        return returns
'''

print(template)
print("\n✅ Use this template for all RL projects!")

## Conclusion

### What We Learned

1. **Gymnasium is the standard RL environment interface**
   - `reset()` → Initialize episode
   - `step(action)` → Execute action
   - `render()` → Visualize

2. **Observation and action spaces** define what agents perceive and control
   - Discrete: Finite set of options
   - Continuous: Unbounded real values
   - Box: Multi-dimensional bounded regions

3. **Custom environments** can be built by inheriting from `gym.Env` and implementing the API

4. **Visualization is crucial** for understanding agent behavior
   - Learning curves track progress
   - State visitation heatmaps show exploration
   - Policy visualization reveals learned strategies

5. **Debugging is systematic** — Use checklists and logging

### Next Steps

You now have the tools to:
- Use any Gymnasium environment
- Build custom environments
- Visualize and debug learning
- Structure professional RL code

In Lesson 1, we'll learn about **Markov Decision Processes (MDPs)** — the mathematical foundation underlying all RL algorithms. We'll see how to formally define problems so RL algorithms can solve them optimally. 🚀